In [1]:
import os
import shutil
import yaml
from sklearn.model_selection import KFold
from ultralytics import YOLO

In [2]:
DATASET_DIR = "../dataset"
FOLDS_DIR = "folds"
os.makedirs(FOLDS_DIR, exist_ok=True)

with open(os.path.join(DATASET_DIR, "data.yaml")) as f:
    data_cfg = yaml.safe_load(f)

def load_images(folder):
    return [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".jpg")]

def get_image_label_pairs(image_paths):
    pairs = []
    for img_path in image_paths:
        label_path = img_path.replace("images", "labels").replace(".jpg", ".txt")
        pairs.append((img_path, label_path))
    return pairs

train_images = load_images(os.path.join(DATASET_DIR, "train/images"))
valid_images = load_images(os.path.join(DATASET_DIR, "valid/images"))

all_image_label_pairs = get_image_label_pairs(train_images + valid_images)

all_images = train_images + valid_images

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [3]:
fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(all_images), 1):
    print(f"\n=== Fold {fold_idx} ===")

    fold_path = os.path.join(FOLDS_DIR, f"fold_{fold_idx}")
    train_dir = os.path.join(fold_path, "images/train")
    val_dir = os.path.join(fold_path, "images/val")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    train_label_dir = os.path.join(fold_path, "labels/train")
    val_label_dir = os.path.join(fold_path, "labels/val")
    os.makedirs(train_label_dir, exist_ok=True)
    os.makedirs(val_label_dir, exist_ok=True)

    for i in train_idx:
        image_path = all_images[i]
        label_path = image_path.replace("images", "labels").replace(".jpg", ".txt")

        dest_image = os.path.join(train_dir, os.path.basename(image_path))
        dest_label = os.path.join(train_label_dir, os.path.basename(label_path))

        if not os.path.exists(dest_image):
            shutil.copy(image_path, dest_image)
        if os.path.exists(label_path):
            shutil.copy(label_path, dest_label)

    for i in val_idx:
        image_path = all_images[i]
        label_path = image_path.replace("images", "labels").replace(".jpg", ".txt")

        dest_image = os.path.join(val_dir, os.path.basename(image_path))
        dest_label = os.path.join(val_label_dir, os.path.basename(label_path))

        if not os.path.exists(dest_image):
            shutil.copy(image_path, dest_image)
        if os.path.exists(label_path):
            shutil.copy(label_path, dest_label)

    # Create a new data.yaml for this fold
    fold_data_yaml = os.path.join(fold_path, "data.yaml")
    data_cfg_fold = data_cfg.copy()
    
    # Use absolute paths for train and val directories
    data_cfg_fold["train"] = os.path.abspath(train_dir)
    data_cfg_fold["val"] = os.path.abspath(val_dir)

    # Print the number of images in each fold
    print(f"Train images in fold {fold_idx}: {len(os.listdir(train_dir))}")
    print(f"Validation images in fold {fold_idx}: {len(os.listdir(val_dir))}")

    with open(fold_data_yaml, "w") as f:
        yaml.dump(data_cfg_fold, f)

    # Print the data.yaml path for debugging
    print(f"Created data.yaml: {fold_data_yaml}")

    # Train YOLO
    model = YOLO("yolov8n.pt")  # Ensure yolov8n.pt is in the correct directory
    results = model.train(
        data=fold_data_yaml,
        epochs=20,
        imgsz=640,
        batch=16,
        plots=True,
        resume=False,  # Set resume=False to train from scratch for each fold
        workers=0
    )

    # Collect metrics for this fold
    metrics = results.results_dict
    fold_metrics.append(metrics)

# Summarize results from all folds
print("\n===== K-FOLD RESULTS =====")
for i, m in enumerate(fold_metrics, 1):
    print(f"Fold {i}: {m}")


=== Fold 1 ===
Train images in fold 1: 1066
Validation images in fold 1: 267
Created data.yaml: folds\fold_1\data.yaml
New https://pypi.org/project/ultralytics/8.3.205 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.202  Python-3.12.1 torch-2.8.0+cu129 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=folds\fold_1\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_de